# LiDAR exploration: building context

This notebook validates classifications, spatial coverage and ground-normalized heights before choosing a reconstruction method. It consumes a small derived crop; it never downloads or stores the complete CNIG tile.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import laspy
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go

from urbanstock3d.processors.lidar import points_in_polygon
from urbanstock3d.providers.pnoa_lidar import footprint_rings_utm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BUILDING_ID = 'ES.SDGC.BU.4531917YJ2743B'
CROP_PATH = PROJECT_ROOT / 'outputs' / BUILDING_ID / 'lidar_context_crop.laz'
BUILDING_PATH = PROJECT_ROOT / 'outputs' / BUILDING_ID / 'building.geojson'
assert CROP_PATH.exists(), f'Generate the crop first: {CROP_PATH}'
assert BUILDING_PATH.exists(), f'Missing building geometry: {BUILDING_PATH}'

CLASS_STYLES = {
    1: ('Unclassified', '#7f7f7f'),
    2: ('Ground', '#8c564b'),
    3: ('Low vegetation', '#bcbd22'),
    4: ('Medium vegetation', '#2ca02c'),
    5: ('High vegetation', '#006400'),
    6: ('Building', '#d62728'),
    7: ('Noise', '#9467bd'),
    12: ('Legacy overlap', '#17becf'),
}
DEFAULT_CLASS_STYLE = ('Other', '#000000')
HEIGHT_COLORMAP = 'viridis'

## 1. Load the reproducible context crop

In [ ]:
cloud = laspy.read(CROP_PATH)
x = np.asarray(cloud.x)
y = np.asarray(cloud.y)
z = np.asarray(cloud.z)
classification = np.asarray(cloud.classification, dtype=np.uint8)

print(f'Points: {len(cloud.points):,}')
print(f'CRS: {cloud.header.parse_crs()}')
print(f'Point format: {cloud.header.point_format.id}')
print(f'Dimensions: {list(cloud.point_format.dimension_names)}')
Counter(classification)

## 2. Inspect classifications in plan view

Class 12 is a legacy overlap classification in this tile, so it is visualized but excluded from geometric measurements.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
for class_id in np.unique(classification):
    mask = classification == class_id
    label, color = CLASS_STYLES.get(int(class_id), DEFAULT_CLASS_STYLE)
    ax.scatter(x[mask], y[mask], s=1, c=color, label=f'{label} ({mask.sum():,})')
ax.set(title='PNOA-LiDAR classifications', xlabel='Easting (m)', ylabel='Northing (m)')
ax.set_aspect('equal')
ax.legend(markerscale=5, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

## 3. Compare usable points and legacy overlap

In [ ]:
usable = classification != 12
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for class_id in np.unique(classification[usable]):
    mask = usable & (classification == class_id)
    label, color = CLASS_STYLES.get(int(class_id), DEFAULT_CLASS_STYLE)
    axes[0].scatter(x[mask], y[mask], c=color, s=2, label=label)
axes[0].set_title(f'Usable points ({usable.sum():,})')
axes[0].legend(markerscale=4)
overlap_label, overlap_color = CLASS_STYLES[12]
axes[1].scatter(x[~usable], y[~usable], c=overlap_color, s=2)
axes[1].set_title(f'{overlap_label} — class 12 ({(~usable).sum():,})')
for ax in axes:
    ax.set_aspect('equal')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
plt.show()

## 4. Normalize elevation against nearby classified ground

In [ ]:
ground = classification == 2
ground_z = float(np.median(z[ground]))
height = z - ground_z
print(f'Ground reference (median class 2): {ground_z:.2f} m')

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
axes[0].hist(height[usable], bins=80, color='#7f7f7f')
axes[0].set(title='Ground-normalized height distribution', xlabel='Height (m)', ylabel='Points')
for class_id in np.unique(classification[usable]):
    mask = usable & (classification == class_id)
    label, color = CLASS_STYLES.get(int(class_id), DEFAULT_CLASS_STYLE)
    axes[1].scatter(x[mask], height[mask], c=color, s=2, label=label)
axes[1].set(title='East-height cross-section', xlabel='Easting (m)', ylabel='Height (m)')
axes[1].legend(markerscale=4)
plt.show()

## 5. Isolate classified roof points inside the cadastral footprint

In [ ]:
building = json.loads(BUILDING_PATH.read_text(encoding='utf-8'))
rings = footprint_rings_utm(building['geometry']['coordinates'])
in_footprint = points_in_polygon(x, y, rings)
roof = in_footprint & (classification == 6)
roof_height = height[roof]

print(f'Classified roof points in footprint: {roof.sum():,}')
print(f'Roof height p50: {np.percentile(roof_height, 50):.2f} m')
print(f'Roof height p95: {np.percentile(roof_height, 95):.2f} m')

fig, ax = plt.subplots(figsize=(9, 7))
plot = ax.scatter(x[roof], y[roof], c=roof_height, s=8, cmap=HEIGHT_COLORMAP)
ax.set(title='Roof points inside cadastral footprint', xlabel='Easting (m)', ylabel='Northing (m)')
ax.set_aspect('equal')
fig.colorbar(plot, ax=ax, label='Height above ground (m)')
plt.show()

## 6. Explore the point cloud interactively in 3D

The cloud is deterministically downsampled when necessary to keep interaction responsive. Each classification uses the same categorical color as the 2D views, while the vertical axis shows height above the local ground reference. Use the legend to show or hide individual classes.

In [ ]:
# Limit the displayed points to keep rotation and zoom responsive.
max_points = 30_000
indices = np.flatnonzero(usable)

if len(indices) > max_points:
    rng = np.random.default_rng(42)
    indices = rng.choice(indices, max_points, replace=False)

figure = go.Figure()
for class_id in np.unique(classification[indices]):
    class_indices = indices[classification[indices] == class_id]
    label, color = CLASS_STYLES.get(int(class_id), DEFAULT_CLASS_STYLE)
    figure.add_trace(
        go.Scatter3d(
            x=x[class_indices],
            y=y[class_indices],
            z=height[class_indices],
            mode='markers',
            name=f'{label} — class {class_id}',
            marker={'size': 1.5, 'color': color, 'opacity': 0.8},
            customdata=np.full(len(class_indices), class_id),
            hovertemplate=(
                'Easting: %{x:.2f} m<br>'
                'Northing: %{y:.2f} m<br>'
                'Height: %{z:.2f} m<br>'
                'Class: %{customdata}<extra>%{fullData.name}</extra>'
            ),
        )
    )

figure.update_layout(
    title='Ground-normalized LiDAR point cloud',
    scene={
        'xaxis_title': 'Easting (m)',
        'yaxis_title': 'Northing (m)',
        'zaxis_title': 'Height above ground (m)',
        'aspectmode': 'data',
    },
    legend={'title': 'PNOA-LiDAR class'},
    height=750,
)

figure.show()

## Observations

Record conclusions here after running the cells:

- Is class 12 spatially redundant with the usable coverage?
- Are class-6 points clean inside the cadastral footprint?
- Does the ground median represent the local terrain adequately?
- Are multiple roof-height modes visible?
- Which artifacts should be removed before estimating normals and planes?